Setup

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 30)
pd.set_option('display.width',140)
plt.rcParams['figure.figsize'] = (9, 4.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

print('pandas', pd.__version__)
print('numpy', np.__version__)

pandas 3.0.6
numpy 2.2.6


load data

In [2]:
df = pd.read_csv('loan_data.csv')
df.shape


(10038, 16)

In [3]:
df.head()

,customer_id,age,monthly_income,employment_years,credit_score,num_dependents,existing_loans,loan_amount,loan_term_months,interest_rate,collateral_value,branch_code,district,product_type,defaulted,days_past_due_final
0,BA101507,36.0,50600.0,14.1,476.0,2,1,1356000,48,10.71,154000.0,138,Chattogram,Working Capital,0,0
1,BA108210,33.0,NaN,NaN,512.0,4,0,225000,24,11.49,304000.0,118,Chattogram,Working Capital,0,5
2,BA101455,35.0,51800.0,NaN,758.0,2,2,1147000,60,13.64,1971000.0,123,Dhaka,Working Capital,0,0
3,BA105323,51.0,119000.0,5.2,614.0,1,1,3510000,24,14.18,6509000.0,133,Dhaka,Working Capital,0,0
4,BA108724,40.0,44900.0,NaN,644.0,1,1,411000,24,13.42,510000.0,124,Chattogram,Working Capital,0,0


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10038 entries, 0 to 10037
Data columns (total 16 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   customer_id          10038 non-null  str    
 1   age                  9957 non-null   float64
 2   monthly_income       8215 non-null   float64
 3   employment_years     6891 non-null   float64
 4   credit_score         9064 non-null   float64
 5   num_dependents       10038 non-null  int64  
 6   existing_loans       10038 non-null  int64  
 7   loan_amount          10038 non-null  int64  
 8   loan_term_months     10038 non-null  int64  
 9   interest_rate        10038 non-null  float64
 10  collateral_value     9427 non-null   float64
 11  branch_code          10038 non-null  int64  
 12  district             10038 non-null  str    
 13  product_type         10038 non-null  str    
 14  defaulted            10038 non-null  int64  
 15  days_past_due_final  10038 non-null  int64  
dt

In [5]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
age,9957.0,4.135724e+01,1.126268e+01,21.0,34.00,41.00,48.00,220.0
monthly_income,8215.0,6.240596e+04,3.808999e+04,0.0,36800.00,53600.00,77600.00,850800.0
employment_years,6891.0,6.193412e+00,3.825281e+00,0.1,3.40,5.40,8.20,38.9
credit_score,9064.0,6.596129e+02,8.384469e+01,370.0,604.00,659.00,718.00,850.0
num_dependents,10038.0,1.807133e+00,1.361251e+00,0.0,1.00,2.00,3.00,8.0
existing_loans,10038.0,8.974895e-01,9.370713e-01,0.0,0.00,1.00,1.00,6.0
loan_amount,10038.0,1.285845e+06,4.091150e+06,59000.0,599000.00,997000.00,1607000.00,400000000.0
loan_term_months,10038.0,3.646025e+01,1.827016e+01,12.0,24.00,36.00,48.00,84.0
interest_rate,10038.0,1.178123e+01,1.898970e+00,7.0,10.47,11.77,13.08,18.0
collateral_value,9427.0,1.195101e+06,1.231785e+06,0.0,362000.00,821000.00,1606000.00,13149000.0


In [6]:
df.isnull().sum().sort_values(ascending=False)

employment_years       3147
monthly_income         1823
credit_score            974
collateral_value        611
age                      81
customer_id               0
existing_loans            0
num_dependents            0
loan_amount               0
loan_term_months          0
interest_rate             0
branch_code               0
district                  0
product_type              0
defaulted                 0
days_past_due_final       0
dtype: int64

In [7]:
df['defaulted'].value_counts()

defaulted
0    9198
1     840
Name: count, dtype: int64

In [8]:
df['defaulted'].value_counts(normalize=True).round(4) * 100

defaulted
0    91.63
1     8.37
Name: proportion, dtype: float64

In [9]:
df.groupby('defaulted')['days_past_due_final'].describe()[['mean','min','max']]

,mean,min,max
defaulted,,,
0,7.645684,0.0,30.0
1,248.554762,92.0,399.0


In [10]:
LEAKY = ['days_past_due_final']
ID_COLS = ['customer_id']
TARGET = 'defaulted'
df = df.drop(columns = LEAKY + ID_COLS)
print('columns remaining:', list(df.columns))

columns remaining: ['age', 'monthly_income', 'employment_years', 'credit_score', 'num_dependents', 'existing_loans', 'loan_amount', 'loan_term_months', 'interest_rate', 'collateral_value', 'branch_code', 'district', 'product_type', 'defaulted']


In [11]:
print('duplicate rows:', df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)
print('shape after dropping duplicates:', df.shape)

duplicate rows: 38
shape after dropping duplicates: (10000, 14)


In [12]:
df.loc[df['age'] > 100, 'age'] = np.nan
df.loc[df['monthly_income'] == 0, 'monthly_income'] = np.nan

df['branch_code'] = df['branch_code'].astype('string')
df[['age', 'monthly_income']].describe().T

,count,mean,std,min,25%,50%,75%,max
age,9911.0,41.196650,9.902857,21.0,34.0,41.0,48.0,74.0
monthly_income,8176.0,62464.077789,38088.540981,6700.0,36900.0,53600.0,77600.0,850800.0


In [13]:
from numpy import median


num_cols = df.select_dtypes(include='number').columns.drop(TARGET)

for col in num_cols:
    if df[col].isnull().any():
        median = df[col].median()
        df[col] = df[col].fillna(median)
        print(f"{col:<20} filled with median = {median:,.0f}")
print('\nRemaining nulls:', int(df.isnull().sum().sum()))

age                  filled with median = 41
monthly_income       filled with median = 53,600
employment_years     filled with median = 5
credit_score         filled with median = 659
collateral_value     filled with median = 821,000

Remaining nulls: 0


In [14]:
x = df.drop(columns=[TARGET])
y = df[TARGET]

x = pd.get_dummies(x, columns=['district','product_type','branch_code'], drop_first=True)
print('x shape:', x.shape)
print('y shape: ', y.shape)
x.head(3)

x shape: (10000, 79)
y shape:  (10000,)


,age,monthly_income,employment_years,credit_score,num_dependents,existing_loans,loan_amount,loan_term_months,interest_rate,collateral_value,district_Chattogram,district_Dhaka,district_Khulna,district_Rajshahi,district_Rangpur,...,branch_code_146,branch_code_147,branch_code_148,branch_code_149,branch_code_150,branch_code_151,branch_code_152,branch_code_153,branch_code_154,branch_code_155,branch_code_156,branch_code_157,branch_code_158,branch_code_159,branch_code_160
0,36.0,50600.0,14.1,476.0,2,1,1356000,48,10.71,154000.0,True,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
1,33.0,53600.0,5.4,512.0,4,0,225000,24,11.49,304000.0,True,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
2,35.0,51800.0,5.4,758.0,2,2,1147000,60,13.64,1971000.0,False,True,False,False,False,...,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False


In [15]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(x, y,
                                                    test_size=0.2,
                                                    random_state=42,
                                                    stratify=y)
print(f'training rows: {len(x_train):,} default rate: {y_train.mean():.2%}')
print(f'test rows: {len(x_test):,} default rate: {y_test.mean():.2%}')

training rows: 8,000 default rate: 8.34%
test rows: 2,000 default rate: 8.35%


In [16]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

model = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(random_state=42, max_iter=1000))
])
model.fit(x_train, y_train)
print('trained...')

trained...


In [17]:
predictions = model.predict(x_test)
print('first 15 hard decisions:', predictions[:15])
print('0 = will repay, 1 = will default')

first 15 hard decisions: [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
0 = will repay, 1 = will default


In [18]:
probabilities = model.predict_proba(x_test)[:, 1]
print('first 8 probabilities of default:', np.round(probabilities[:8], 4))

first 8 probabilities of default: [0.1149 0.0283 0.2439 0.0142 0.0858 0.0041 0.017  0.0038]


In [19]:
pd.DataFrame({
    'p(default)': probabilities[:10].round(4),
    'predicted' : predictions[:10],
    'actual': y_test.values[:10]
})

,p(default),predicted,actual
0,0.1149,0,0
1,0.0283,0,0
2,0.2439,0,1
3,0.0142,0,0
4,0.0858,0,0
5,0.0041,0,0
6,0.0170,0,0
7,0.0038,0,0
8,0.0020,0,0
9,0.0136,0,0


In [20]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score

baseline = DummyClassifier(strategy='most_frequent').fit(x_train, y_train)
print(f'model accuracy: {accuracy_score(y_test, predictions):.2%}')
print(f'baseline accuracy: {accuracy_score(y_test, baseline.predict(x_test)):.2%}')
print("\n the baseline predicts 'no default' for every single application ")

model accuracy: 92.25%
baseline accuracy: 91.65%

 the baseline predicts 'no default' for every single application 


In [21]:
from sklearn.metrics import classification_report, confusion_matrix

cm = confusion_matrix(y_test, predictions)
tn, fp, fn, tp = cm.ravel()

print(pd.DataFrame(cm,
                   index=['actually repaid', 'actually defaulted'],
                   columns=['predicted repaid', 'predicted defaulted']))

print(f"\n missed defaulters (false negatives): {fn}")
print(f'good customers incorrectly predicted to default (false positives): {fp}')

                    predicted repaid  predicted defaulted
actually repaid                 1823                   10
actually defaulted               145                   22

 missed defaulters (false negatives): 145
good customers incorrectly predicted to default (false positives): 10


In [22]:
print(classification_report(y_test, predictions, target_names=['repaid', 'defaulted'],digits=3))

              precision    recall  f1-score   support

      repaid      0.926     0.995     0.959      1833
   defaulted      0.688     0.132     0.221       167

    accuracy                          0.922      2000
   macro avg      0.807     0.563     0.590      2000
weighted avg      0.906     0.922     0.898      2000

